# 08 - Build and Verify Database

## Purpose
Load all cleaned datasets from `data/clean/`, design the SQLite schema,
create the database, load the tables, and run verification queries to
confirm everything landed correctly and joins work as expected.

## Inputs
- `data/clean/country_crosswalk.csv`
- `data/clean/cbam_defaults_clean.csv`
- `data/clean/eu_import_trade_flows_clean.csv`
- `data/clean/country_grid_electricity_clean.csv`
- `data/clean/hydrogen_route_intensities_clean.csv`
- `data/clean/steel_route_intensity_clean.csv`

## Output
- `db/cbam.db` — SQLite database

## Schema
- `country_crosswalk` — canonical country identifier mapping
- `cbam_defaults` — CBAM default emission values by country and CN code
- `trade_flows` — EU27 import flows by partner, CN code, and year
- `grid_electricity` — Ember grid CO2 intensity and generation by country and year
- `hydrogen_intensities` — JRC hydrogen route emission intensities
- `steel_route_intensities` — Worldsteel production route emission intensities

## Notes
- `country` (canonical English name) is the join key across country-level tables.
- `iso3` is retained in the crosswalk as a secondary identifier.
- The database is not committed to GitHub. It is generated from this notebook
  and the clean CSVs in `data/clean/`.
- Verification queries go beyond row counts — they test join integrity and
  flag any analytical gaps before the calculation layer is built.

In [2]:
# Imports, paths, and database connection setup.
# The db/ directory is created if it doesn't exist.
import sqlite3
import pandas as pd
from pathlib import Path

clean = Path("../data/clean")
db_path = Path("../db/cbam.db")
db_path.parent.mkdir(exist_ok=True)

# Load all clean datasets
crosswalk   = pd.read_csv(clean / "country_crosswalk.csv")
defaults    = pd.read_csv(clean / "cbam_defaults_clean.csv")
flows       = pd.read_csv(clean / "eu_import_trade_flows_clean.csv")
grid        = pd.read_csv(clean / "country_grid_electricity_clean.csv")
hydrogen    = pd.read_csv(clean / "hydrogen_route_intensities_clean.csv")
steel       = pd.read_csv(clean / "steel_route_intensity_clean.csv")

datasets = {
    "country_crosswalk":      crosswalk,
    "cbam_defaults":          defaults,
    "trade_flows":            flows,
    "grid_electricity":       grid,
    "hydrogen_intensities":   hydrogen,
    "steel_route_intensities":steel,
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

country_crosswalk: (240, 5)
cbam_defaults: (10671, 11)
trade_flows: (155848, 10)
grid_electricity: (193936, 10)
hydrogen_intensities: (6, 5)
steel_route_intensities: (12, 4)


In [3]:
# Rename trade_flows columns for consistency with other tables before loading.
# product -> cn_code (matches cbam_defaults and hydrogen_intensities)
# partner_country -> country (matches cbam_defaults and grid tables)
# partner -> iso2 (reflects what the column actually contains)
flows = flows.rename(columns={
    "product":         "cn_code",
    "partner_country": "country",
    "partner":         "iso2",
})

print("Renamed columns:")
print(flows.columns.tolist())

Renamed columns:
['freq', 'reporter', 'iso2', 'cn_code', 'flow', 'indicator', 'year', 'value', 'material', 'country']


In [4]:
# Split the long-format grid electricity table into three purpose-built tables.
# CO2 intensity is the primary variable for CBAM indirect emissions calculations.
# Capacity and generation are retained for trend analysis and dashboard use.

grid_co2_intensity = grid[
    (grid["Category"] == "Power sector emissions") &
    (grid["Variable"] == "CO2 intensity")
][["country", "ISO 3 code", "Year", "Continent", "Ember region", "Value"]].copy()
grid_co2_intensity = grid_co2_intensity.rename(columns={
    "ISO 3 code":   "iso3",
    "Year":         "year",
    "Ember region": "ember_region",
    "Value":        "co2_intensity_gco2_kwh",
})

grid_capacity = grid[
    grid["Category"] == "Capacity"
][["country", "ISO 3 code", "Year", "Subcategory", "Variable", "Unit", "Value"]].copy()
grid_capacity = grid_capacity.rename(columns={
    "ISO 3 code": "iso3",
    "Year":       "year",
    "Subcategory":"subcategory",
    "Variable":   "fuel_type",
    "Unit":       "unit",
    "Value":      "capacity_gw",
})

grid_generation = grid[
    grid["Category"] == "Electricity generation"
][["country", "ISO 3 code", "Year", "Subcategory", "Variable", "Unit", "Value"]].copy()
grid_generation = grid_generation.rename(columns={
    "ISO 3 code": "iso3",
    "Year":       "year",
    "Subcategory":"subcategory",
    "Variable":   "fuel_type",
    "Unit":       "unit",
    "Value":      "value",
})

print(f"grid_co2_intensity: {grid_co2_intensity.shape}")
print(f"grid_capacity:      {grid_capacity.shape}")
print(f"grid_generation:    {grid_generation.shape}")

grid_co2_intensity: (5407, 6)
grid_capacity:      (62141, 7)
grid_generation:    (126388, 7)


In [5]:
# Create the SQLite database and load all tables.
# If the database already exists it is replaced cleanly.
# Primary keys are declared as comments since SQLite does not enforce
# composite primary keys via pandas to_sql — they are enforced via
# the CREATE TABLE statements below where it matters analytically.

conn = sqlite3.connect(db_path)

# Load tables using pandas to_sql for speed, replacing if they exist
crosswalk.to_sql("country_crosswalk",       conn, if_exists="replace", index=False)
defaults.to_sql("cbam_defaults",            conn, if_exists="replace", index=False)
flows.to_sql("trade_flows",                 conn, if_exists="replace", index=False)
grid_co2_intensity.to_sql("grid_co2_intensity", conn, if_exists="replace", index=False)
grid_capacity.to_sql("grid_capacity",       conn, if_exists="replace", index=False)
grid_generation.to_sql("grid_generation",   conn, if_exists="replace", index=False)
hydrogen.to_sql("hydrogen_intensities",     conn, if_exists="replace", index=False)
steel.to_sql("steel_route_intensities",     conn, if_exists="replace", index=False)

print("Tables loaded:")
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(tables["name"].tolist())

Tables loaded:
['country_crosswalk', 'cbam_defaults', 'trade_flows', 'grid_co2_intensity', 'grid_capacity', 'grid_generation', 'hydrogen_intensities', 'steel_route_intensities']
